# Fama MacBeth

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

In [6]:
preference_returns = pd.read_csv("Preference Returns.csv", index_col="date")

In [7]:
sp500 = pd.read_csv("sp500_returns_with_tickers.csv", index_col="date")

In [8]:
preference_returns

,Preference Return
date,
1960-01-29,-0.018005
1960-02-29,0.009621
1960-03-31,0.015161
1960-04-29,0.002178
1960-05-31,0.012071
...,...
2024-08-30,0.003173
2024-09-30,0.002560
2024-10-31,0.011624


In [1]:
cap_data = pd.read_csv("sp500_market_caps.csv", index_col="date")
cap_data.head()
cap_data = cap_data * 1000
cap_data.tail()
len(cap_data.columns)
cap_data_numeric = cap_data.apply(pd.to_numeric, errors='coerce')
cap_data_numeric = cap_data_numeric.fillna(0)
weight_data = cap_data_numeric.div(cap_data_numeric.sum(axis=1), axis=0)
row_sums = weight_data.sum(axis=1)
print(row_sums.head())
weight_data.tail()

NameError: name 'pd' is not defined

In [9]:
sp500

,ACF,ABK,AMT,JAVA,AN,ORCL,MSFT,SDS,AYE,TROW,...,CFN,AVGO,VRSK,DG.2,FTNT,VAL,GNRC,QEP,CBOE,TSLA
date,,,,,,,,,,,,,,,,,,,,,
1960-01-29,0.005155,-0.015000,-0.020785,NaN,-0.092262,NaN,NaN,NaN,-0.020833,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1960-02-29,0.046154,0.017767,0.009434,NaN,-0.021312,NaN,NaN,NaN,0.053192,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1960-03-31,-0.059553,-0.100249,-0.047170,NaN,0.038851,NaN,NaN,NaN,-0.005387,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1960-04-29,-0.081794,-0.056180,0.000000,NaN,-0.050407,NaN,NaN,NaN,0.010274,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1960-05-31,0.048851,-0.029762,-0.034654,NaN,0.029110,NaN,NaN,NaN,0.044068,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-08-30,NaN,NaN,NaN,NaN,NaN,0.013195,-0.001100,NaN,NaN,-0.071535,...,NaN,0.013319,0.042292,-0.310823,0.321675,NaN,0.005460,NaN,0.122718,-0.077391
2024-09-30,NaN,NaN,NaN,NaN,NaN,0.206030,0.031548,NaN,NaN,0.038948,...,NaN,0.062707,-0.016385,0.019284,0.010950,NaN,0.015013,NaN,-0.002580,0.221942
2024-10-31,NaN,NaN,NaN,NaN,NaN,-0.012676,-0.055659,NaN,NaN,0.008538,...,NaN,-0.015826,0.025228,-0.046589,0.014313,NaN,0.041981,NaN,0.042466,-0.045025


In [10]:
combined = sp500.join(preference_returns, how="outer")   # outer keeps every date

In [11]:
combined

,ACF,ABK,AMT,JAVA,AN,ORCL,MSFT,SDS,AYE,TROW,...,AVGO,VRSK,DG.2,FTNT,VAL,GNRC,QEP,CBOE,TSLA,Preference Return
date,,,,,,,,,,,,,,,,,,,,,
1960-01-29,0.005155,-0.015000,-0.020785,NaN,-0.092262,NaN,NaN,NaN,-0.020833,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.018005
1960-02-29,0.046154,0.017767,0.009434,NaN,-0.021312,NaN,NaN,NaN,0.053192,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.009621
1960-03-31,-0.059553,-0.100249,-0.047170,NaN,0.038851,NaN,NaN,NaN,-0.005387,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.015161
1960-04-29,-0.081794,-0.056180,0.000000,NaN,-0.050407,NaN,NaN,NaN,0.010274,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.002178
1960-05-31,0.048851,-0.029762,-0.034654,NaN,0.029110,NaN,NaN,NaN,0.044068,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.012071
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-08-30,NaN,NaN,NaN,NaN,NaN,0.013195,-0.001100,NaN,NaN,-0.071535,...,0.013319,0.042292,-0.310823,0.321675,NaN,0.005460,NaN,0.122718,-0.077391,0.003173
2024-09-30,NaN,NaN,NaN,NaN,NaN,0.206030,0.031548,NaN,NaN,0.038948,...,0.062707,-0.016385,0.019284,0.010950,NaN,0.015013,NaN,-0.002580,0.221942,0.002560
2024-10-31,NaN,NaN,NaN,NaN,NaN,-0.012676,-0.055659,NaN,NaN,0.008538,...,-0.015826,0.025228,-0.046589,0.014313,NaN,0.041981,NaN,0.042466,-0.045025,0.011624


In [12]:
FACTOR_COL = "Preference Return"
MIN_OBS = 10

In [13]:
# first pass:

asset_cols = combined.columns.drop(FACTOR_COL)

betas, alphas, r2 = {}, {}, {}

In [14]:
for asset in asset_cols:
    ts_data = combined[[asset, FACTOR_COL]].dropna()
    if len(ts_data) < MIN_OBS:
        continue

    y = ts_data[asset]
    X = sm.add_constant(ts_data[FACTOR_COL]) 
    res = sm.OLS(y, X).fit()

    alphas[asset] = res.params["const"]
    betas[asset]  = res.params[FACTOR_COL]
    r2[asset]     = res.rsquared


In [15]:
beta_series  = pd.Series(betas,  name="beta")
alpha_series = pd.Series(alphas, name="alpha")
print(f"Estimated betas for {len(beta_series)} assets.")

Estimated betas for 1826 assets.


In [16]:
beta_series

ACF    -1.349739
ABK    -1.947404
AMT    -2.638726
JAVA   -0.334922
AN     -1.725335
          ...   
VAL    -3.775680
GNRC   -1.069973
QEP    -5.038467
CBOE   -0.803763
TSLA    2.089685
Name: beta, Length: 1826, dtype: float64

In [18]:
## second pass

lambda_vals, lambda_dates = [], []

for date, row in combined.iterrows():

    tradable = row[asset_cols].dropna().index.intersection(beta_series.index)
    if len(tradable) < 5:             # need enough cross-section
        continue

    y_cs = row[tradable].values
    X_cs = sm.add_constant(beta_series[tradable].values)
    res_cs = sm.OLS(y_cs, X_cs).fit()

    lambda_vals.append(res_cs.params[1])          # slope coefficient
    lambda_dates.append(date)

In [19]:
lambda_series = pd.Series(lambda_vals, index=lambda_dates, name="lambda_t")
print(f"Estimated λ_t for {len(lambda_series)} time periods.")

Estimated λ_t for 780 time periods.


In [20]:
mean_lambda = lambda_series.mean()
std_lambda  = lambda_series.std(ddof=1)
t_stat      = mean_lambda / (std_lambda / np.sqrt(len(lambda_series)))

print("\n----- Fama–MacBeth summary -----")
print(f"Mean λ (risk premium):  {mean_lambda: .6f}")
print(f"T-statistic:            {t_stat: .3f}")


----- Fama–MacBeth summary -----
Mean λ (risk premium):   0.001101
T-statistic:             1.743
